<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/paper_summary/02_SLCP_JANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 — JANA-paper and separate-flow baselines

This notebook produces two deliberately distinct baselines.  **JANA-paper**
runs the pinned algorithm and hyperparameters in its isolated legacy
TensorFlow/BayesFlow environment, with only the simulator input adapted to the
fixed nested banks.  In keeping with upstream, this row uses N training pairs,
the two-row shape bank, the fixed two-row Trainer pilot, and the fixed 300-pair
validation bank, and is labelled N+304 in resource tables.  **Separate flows** loads the nominal
posterior and likelihood ensembles selected in notebook 01.

Both baselines receive the full paired diagnostic suite: posterior-route and
likelihood-route C2ST/MMD, route agreement, predictive closure, importance
efficiency, Bayes-cycle and conditional-normalization checks, and comparison
with the analytic SLCP likelihood.  Each is the direct control for corrections
trained over that same flow base in notebook 03.

Several independent Colab runtimes may run this notebook against the same
Drive artifact root.  Each runtime claims one pending `(budget, ML seed)` shard
at a time and skips shards already being trained by another live runtime.


In [1]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization in this process.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("PAPER_SUMMARY_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

def installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP"
        )
    else:
        default_artifact_root = Path("/content/paper_summary_SLCP_artifacts")

    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, repository, env=clone_env,
        )
    else:
        run("git", "-C", repository, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", repository, "fetch", "origin", BRANCH)
        run("git", "-C", repository, "checkout", BRANCH)
        run("git", "-C", repository, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", repository, "sparse-checkout", "set", "src",
        "workshops/ml4hep_tifr_colab/paper_summary",
    )
    SOURCE_DIR = repository / "workshops" / "ml4hep_tifr_colab" / "paper_summary"

    # Colab already provides the numerical/ML stack used by these notebooks.
    # Install only the two missing modern-runtime packages normally.  In
    # particular, do not let sbibm pull its historical algorithm dependency
    # tree into the current Colab Python environment (currently Python 3.13).
    modern_requirements = []
    if installed_version("nflows") != "0.14":
        modern_requirements.append("nflows==0.14")
    if importlib.util.find_spec("pyro") is None:
        modern_requirements.append("pyro-ppl")
    if modern_requirements:
        run(sys.executable, "-m", "pip", "install", "-q", *modern_requirements)
    if installed_version("sbibm") != "1.1.0":
        # This is the same Python-3.13-safe installation used by Exercises 9
        # and 10: the SLCP task/metrics need sbibm itself, nflows, and Pyro,
        # but not sbibm's old pinned SBI/algorithm environment.
        run(
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "sbibm==1.1.0",
        )
else:
    candidates = (
        Path.cwd(),
        Path.cwd() / "paper_summary",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab" / "paper_summary",
    )
    SOURCE_DIR = next(
        (candidate.resolve() for candidate in candidates if (candidate / "config.py").is_file()),
        None,
    )
    if SOURCE_DIR is None:
        raise FileNotFoundError("Cannot locate the paper_summary source directory")
    default_artifact_root = SOURCE_DIR / "artifacts"

source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(SOURCE_DIR)

ARTIFACT_ROOT = Path(
    os.environ.get("PAPER_SUMMARY_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Paper-summary source:", SOURCE_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)


Mounted at /content/drive
Paper-summary source: /content/nsbi-lhc-toolkit/workshops/ml4hep_tifr_colab/paper_summary
Persistent artifact root: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP


In [2]:
from config import (
    DEFAULT_ML_SEEDS,
    PAPER_BUDGETS,
    SMOKE_BUDGETS,
    SMOKE_ML_SEEDS,
    campaign_config,
    campaign_signature,
)

PROFILE = os.environ.get("PAPER_SUMMARY_PROFILE", "PAPER").upper()
CAMPAIGN_BUDGETS = list(PAPER_BUDGETS if PROFILE == "PAPER" else SMOKE_BUDGETS)
CAMPAIGN_ML_SEEDS = list(DEFAULT_ML_SEEDS if PROFILE == "PAPER" else SMOKE_ML_SEEDS)

def execution_subset(environment_name, configured):
    raw = os.environ.get(environment_name, "").strip()
    values = list(configured) if not raw else [int(value) for value in raw.split(",")]
    unknown = set(values) - set(configured)
    if not values or unknown:
        raise ValueError(f"Invalid {environment_name}: {values}; unknown={sorted(unknown)}")
    return values

BUDGETS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_BUDGETS", CAMPAIGN_BUDGETS)
ML_SEEDS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_SEEDS", CAMPAIGN_ML_SEEDS)
LOAD_IF_AVAILABLE = os.environ.get("PAPER_SUMMARY_LOAD_IF_AVAILABLE", "1") != "0"

CAMPAIGN = campaign_config(profile=PROFILE)
print(json.dumps({
    "profile": PROFILE,
    "campaign_budgets": CAMPAIGN_BUDGETS,
    "campaign_ml_seeds": CAMPAIGN_ML_SEEDS,
    "budgets_to_run": BUDGETS_TO_RUN,
    "ml_seeds_to_run": ML_SEEDS_TO_RUN,
    "load_if_available": LOAD_IF_AVAILABLE,
    "campaign_signature": campaign_signature(CAMPAIGN),
}, indent=2))


{
  "profile": "PAPER",
  "campaign_budgets": [
    10000,
    100000,
    1000000
  ],
  "campaign_ml_seeds": [
    31082026,
    31082027,
    31082028
  ],
  "budgets_to_run": [
    10000,
    100000,
    1000000
  ],
  "ml_seeds_to_run": [
    31082026,
    31082027,
    31082028
  ],
  "load_if_available": true,
  "campaign_signature": "sha256-e455fa167513"
}


In [3]:
RUN_EXACT_JANA_PAPER = True
RUN_NOMINAL_MATCHED = True
INSTALL_EXACT_JANA_ENV_IF_MISSING = (
    os.environ.get("PAPER_SUMMARY_INSTALL_JANA_ENV", "1") != "0"
)


In [4]:
if RUN_EXACT_JANA_PAPER:
    # Reload so rerunning this cell after the setup cell pulls a repository
    # update cannot retain an older helper from the current Colab process.
    import importlib
    import utils_jana
    import utils_jana_runtime

    utils_jana = importlib.reload(utils_jana)
    utils_jana_runtime = importlib.reload(utils_jana_runtime)

    print("Preparing the isolated exact-JANA runtime (first install can take several minutes).")
    JANA_PYTHON = utils_jana_runtime.ensure_jana_environment(
        ARTIFACT_ROOT,
        install_if_missing=INSTALL_EXACT_JANA_ENV_IF_MISSING,
    )
    print("Exact-JANA Python:", JANA_PYTHON)


Preparing the isolated exact-JANA runtime (first install can take several minutes).
Rebuilding isolated exact-JANA environment: /content/paper_summary_jana_env
Installing pinned exact-JANA packages into: /content/paper_summary_jana_env
Exact-JANA installation probe: {"base_prefix": "/root/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu", "executable": "/content/paper_summary_jana_env/bin/python", "numpy": "1.23.5", "prefix": "/content/paper_summary_jana_env", "purelib": "/content/paper_summary_jana_env/lib/python3.11/site-packages", "python": "3.11.16", "site_packages": ["/content/paper_summary_jana_env/lib/python3.11/site-packages"]}
Exact-JANA Python: /content/paper_summary_jana_env/bin/python


In [5]:
from IPython.display import display

def display_result(result):
    if hasattr(result, "style"):
        display(result.style.format(precision=4).hide(axis="index"))
    elif isinstance(result, dict):
        for name, value in result.items():
            print(f"\n{name}")
            if hasattr(value, "style"):
                display(value.style.format(precision=4).hide(axis="index"))
            else:
                display(value)
    else:
        display(result)


In [6]:
import importlib
import utils

# Pull repository fixes into an already-open Colab runtime.
utils = importlib.reload(utils)

JANA_RESULT = utils.run_jana_campaign(
    artifact_root=ARTIFACT_ROOT,
    campaign=CAMPAIGN,
    run_exact_paper=RUN_EXACT_JANA_PAPER,
    run_matched=RUN_NOMINAL_MATCHED,
    budgets_to_run=BUDGETS_TO_RUN,
    ml_seeds_to_run=ML_SEEDS_TO_RUN,
    load_if_available=LOAD_IF_AVAILABLE,
)
display_result(JANA_RESULT)


[exact JANA] Running budget=1000000/seed=31082026


RuntimeError: The isolated exact-JANA evaluation failed. Its trained checkpoint was not modified. Isolated-process diagnostics:
2026-09-07 21:47:11.487549: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-07 21:47:11.489927: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-07 21:47:11.539858: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-07 21:47:11.540343: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the ap